# Advanced Anomaly Detection: Random Walk + LDA + LSTM + Hybrid Score
Phát hiện dị thường đánh giá sử dụng Random Walk trên đồ thị, LDA cho ngữ nghĩa, LSTM cho nội dung, và hybrid scoring

## 1. Setup và Dependencies

In [1]:
import subprocess
import sys

packages = [
    'numpy',
    'pandas',
    'gensim',
    'scikit-learn',
    'scipy',
    'tqdm'
]

print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ All packages installed!")

Installing packages...
✅ All packages installed!


Installing packages...
✅ All packages installed!


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## 2. Import và Setup

In [2]:
import json
import logging
import platform
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import csr_matrix, lil_matrix
from scipy.stats import entropy
from tqdm import tqdm

from gensim.corpora import Dictionary
from gensim.models import LdaModel, Word2Vec
from gensim.utils import simple_preprocess

from sklearn.preprocessing import StandardScaler

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Setup paths
if platform.system() == "Darwin":
    PROJECT_DIR = Path.home() / "Downloads" / "MXH FINAL"
else:
    PROJECT_DIR = Path(".") / "MXH FINAL"

DATA_DIR = PROJECT_DIR / "data"
GOLD_DIR = DATA_DIR / "gold" / "All_Beauty"
HYBRID_DIR = GOLD_DIR / "hybrid_model"
ANOMALY_DIR = GOLD_DIR / "anomaly_detection"
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"PROJECT_DIR: {PROJECT_DIR}")
logger.info(f"HYBRID_DIR: {HYBRID_DIR}")
logger.info(f"ANOMALY_DIR: {ANOMALY_DIR}")

/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026-06-09 13:48:04,364 - INFO - PROJECT_DIR: /Users/mac/Downloads/MXH FINAL
2026-06-09 13:48:04,364 - INFO - HYBRID_DIR: /Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model
2026-06-09 13:48:04,365 - INFO - ANOMALY_DIR: /Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/anomaly_detection


## 3. Load Preprocessed Data từ Hybrid Pipeline

In [3]:
# Load hybrid model outputs
logger.info("Loading hybrid model outputs...")

combined_features = np.load(HYBRID_DIR / "combined_features.npy")
labels = np.load(HYBRID_DIR / "labels.npy")
splits = np.load(HYBRID_DIR / "splits.npy")

with open(HYBRID_DIR / "graph_structure.json") as f:
    graph_structure = json.load(f)

with open(HYBRID_DIR / "id_mappings.json") as f:
    id_mappings = json.load(f)

# Load original data
reviews_clean = pd.read_parquet(GOLD_DIR / "reviews_clean.parquet")
review_features = pd.read_parquet(GOLD_DIR / "review_features.parquet")
edges_user_review = pd.read_parquet(GOLD_DIR / "edges_user_review.parquet")
edges_review_item = pd.read_parquet(GOLD_DIR / "edges_review_item.parquet")
edges_user_item = pd.read_parquet(GOLD_DIR / "edges_user_item.parquet")
edges_review_review = pd.read_parquet(GOLD_DIR / "edges_review_review.parquet")

# Load embeddings
review_embeddings = np.load(GOLD_DIR / "review_text_embeddings_All_Beauty.npy")

logger.info(f"✅ Loaded {combined_features.shape[0]} reviews")
logger.info(f"✅ Feature dimension: {combined_features.shape[1]}")
logger.info(f"✅ Embedding dimension: {review_embeddings.shape[1]}")

2026-06-09 13:48:04,371 - INFO - Loading hybrid model outputs...
2026-06-09 13:48:09,478 - INFO - ✅ Loaded 644689 reviews
2026-06-09 13:48:09,479 - INFO - ✅ Feature dimension: 420
2026-06-09 13:48:09,480 - INFO - ✅ Embedding dimension: 384


## 4. Random Walk + Transition Probability Matrix

In [4]:
class RandomWalkAnomalyDetector:
    """Random Walk-based anomaly detection with transition probability matrix"""
    
    def __init__(self, num_nodes: int, damping_factor: float = 0.85, num_walks: int = 100):
        """
        Args:
            num_nodes: Number of nodes in graph
            damping_factor: Damping factor for PageRank (default 0.85)
            num_walks: Number of random walks per node
        """
        self.num_nodes = num_nodes
        self.damping_factor = damping_factor
        self.num_walks = num_walks
        self.transition_matrix = None
        self.pagerank_scores = None
        self.walk_distributions = None
    
    def build_transition_matrix(self, edges: List[Tuple[int, int]], 
                               edge_weights: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Build transition probability matrix P from edges
        P[i][j] = probability of moving from node i to node j
        """
        P = lil_matrix((self.num_nodes, self.num_nodes))
        
        # Count outgoing edges per node
        out_degree = np.zeros(self.num_nodes)
        
        for idx, (src, dst) in enumerate(edges):
            weight = edge_weights[idx] if edge_weights is not None else 1.0
            P[src, dst] += weight
            out_degree[src] += weight
        
        # Normalize: P[i][j] = P[i][j] / sum(P[i][:])
        for i in range(self.num_nodes):
            if out_degree[i] > 0:
                P[i, :] /= out_degree[i]
            else:
                # Dangling node: uniform distribution
                P[i, :] = 1.0 / self.num_nodes
        
        self.transition_matrix = P.tocsr()
        logger.info(f"✅ Transition matrix built: shape {self.transition_matrix.shape}")
        return self.transition_matrix
    
    def compute_pagerank(self, max_iter: int = 100, tol: float = 1e-6) -> np.ndarray:
        """
        Compute PageRank scores using power iteration
        """
        if self.transition_matrix is None:
            raise ValueError("Transition matrix not built. Call build_transition_matrix first.")
        
        # Initialize with uniform distribution
        r = np.ones(self.num_nodes) / self.num_nodes
        
        for iteration in range(max_iter):
            r_new = self.damping_factor * self.transition_matrix.T @ r + (1 - self.damping_factor) / self.num_nodes
            
            # Check convergence
            if np.linalg.norm(r_new - r) < tol:
                logger.info(f"✅ PageRank converged at iteration {iteration}")
                break
            r = r_new
        
        self.pagerank_scores = r
        return r
    
    def compute_walk_distribution(self, start_node: int, length: int = 10) -> np.ndarray:
        """
        Compute stationary distribution from random walks starting at node
        """
        if self.transition_matrix is None:
            raise ValueError("Transition matrix not built.")
        
        # Multi-step transition probabilities
        prob = np.zeros(self.num_nodes)
        prob[start_node] = 1.0
        
        for step in range(length):
            prob = self.transition_matrix.T @ prob
        
        return prob
    
    def detect_intrusion_anomalies(self, review_ids: List[int], 
                                    threshold_percentile: float = 95.0) -> Dict:
        """
        Detect intrusive/anomalous behavior using PageRank deviation
        """
        if self.pagerank_scores is None:
            self.compute_pagerank()
        
        # Compute anomaly scores as deviation from expected PageRank
        expected_score = np.mean(self.pagerank_scores)
        anomaly_scores = np.abs(self.pagerank_scores - expected_score)
        
        threshold = np.percentile(anomaly_scores, threshold_percentile)
        is_anomaly = anomaly_scores > threshold
        
        return {
            'anomaly_scores': anomaly_scores,
            'is_anomaly': is_anomaly,
            'threshold': threshold,
            'num_anomalies': np.sum(is_anomaly)
        }

logger.info("✅ RandomWalkAnomalyDetector class defined")

2026-06-09 13:48:09,499 - INFO - ✅ RandomWalkAnomalyDetector class defined


## 5. LDA + LSTM Semantic Correlation Extractor

In [5]:
class SemanticCorrelationExtractor:
    """Extract semantic correlations using LDA and Word2Vec"""
    
    def __init__(self, num_topics: int = 10, embedding_dim: int = 64):
        self.num_topics = num_topics
        self.embedding_dim = embedding_dim
        self.lda_model = None
        self.dictionary = None
        self.w2v_model = None
        self.lda_features = None
        self.lstm_features = None  # Reuse for word2vec features
    
    def prepare_corpus(self, texts: List[str], min_freq: int = 2) -> Tuple[Dictionary, List[List[int]], List[List[str]]]:
        """
        Prepare corpus for LDA: tokenize and create dictionary
        """
        logger.info(f"Preparing corpus from {len(texts)} texts...")
        
        # Simple preprocessing and tokenization
        processed_texts = []
        for text in tqdm(texts, desc="Preprocessing texts"):
            tokens = simple_preprocess(str(text), deacc=True, min_len=3)
            processed_texts.append(tokens)
        
        # Create dictionary
        self.dictionary = Dictionary(processed_texts)
        self.dictionary.filter_extremes(no_below=min_freq, no_above=0.7, keep_n=1000)
        
        # Create corpus (bag-of-words)
        corpus = [self.dictionary.doc2bow(text) for text in processed_texts]
        
        logger.info(f"✅ Dictionary size: {len(self.dictionary)}")
        logger.info(f"✅ Corpus size: {len(corpus)}")
        
        return self.dictionary, corpus, processed_texts
    
    def train_lda(self, corpus: List[List[int]], num_iterations: int = 50):
        """
        Train LDA model on corpus
        """
        logger.info(f"Training LDA with {self.num_topics} topics...")
        
        self.lda_model = LdaModel(
            corpus=corpus,
            id2word=self.dictionary,
            num_topics=self.num_topics,
            random_state=42,
            passes=10,
            iterations=num_iterations,
            per_word_topics=True,
            minimum_probability=0.0
        )
        
        logger.info(f"✅ LDA model trained")
        return self.lda_model
    
    def extract_lda_features(self, corpus: List[List[int]]) -> np.ndarray:
        """
        Extract topic distribution for each document
        Returns: (num_docs, num_topics) array
        """
        if self.lda_model is None:
            raise ValueError("LDA model not trained")
        
        lda_features = []
        for doc_bow in tqdm(corpus, desc="Extracting LDA features"):
            doc_topics = self.lda_model.get_document_topics(doc_bow, minimum_probability=0)
            # Convert to dense vector
            topic_dist = np.zeros(self.num_topics)
            for topic_id, prob in doc_topics:
                topic_dist[topic_id] = prob
            lda_features.append(topic_dist)
        
        self.lda_features = np.array(lda_features)
        logger.info(f"✅ LDA features shape: {self.lda_features.shape}")
        return self.lda_features
    
    def build_word2vec_model(self, processed_texts: List[List[str]]):
        """
        Build Word2Vec model for text encoding (lightweight alternative to LSTM)
        """
        logger.info(f"Training Word2Vec with {self.embedding_dim} dimensions...")
        
        self.w2v_model = Word2Vec(
            sentences=processed_texts,
            vector_size=self.embedding_dim,
            window=5,
            min_count=2,
            workers=4,
            sg=1,  # Skip-gram
            seed=42
        )
        
        logger.info(f"✅ Word2Vec model trained")
        logger.info(f"   Vocabulary size: {len(self.w2v_model.wv)}")
        return self.w2v_model
    
    def extract_word2vec_features(self, processed_texts: List[List[str]]) -> np.ndarray:
        """
        Extract Word2Vec embeddings by averaging word vectors per document
        Returns: (num_docs, embedding_dim) array
        """
        if self.w2v_model is None:
            raise ValueError("Word2Vec model not built")
        
        word2vec_features = []
        for tokens in tqdm(processed_texts, desc="Extracting Word2Vec features"):
            # Get vectors for words in vocabulary
            vectors = [self.w2v_model.wv[token] for token in tokens if token in self.w2v_model.wv]
            if vectors:
                # Average word vectors
                doc_vector = np.mean(vectors, axis=0)
            else:
                # OOV documents: zero vector
                doc_vector = np.zeros(self.embedding_dim)
            word2vec_features.append(doc_vector)
        
        self.lstm_features = np.array(word2vec_features)  # Reuse lstm_features for compatibility
        logger.info(f"✅ Word2Vec features shape: {self.lstm_features.shape}")
        return self.lstm_features
    
    def compute_semantic_correlation(self, doc_ids: Optional[List[int]] = None) -> np.ndarray:
        """
        Compute semantic correlation matrix between documents
        Combines LDA topic similarity + Word2Vec embedding similarity
        Returns: (num_docs, num_docs) correlation matrix
        """
        if self.lda_features is None or self.lstm_features is None:
            raise ValueError("LDA and Word2Vec features must be extracted first")
        
        num_docs = len(self.lda_features)
        
        # Normalize features
        lda_norm = self.lda_features / (np.linalg.norm(self.lda_features, axis=1, keepdims=True) + 1e-8)
        w2v_norm = self.lstm_features / (np.linalg.norm(self.lstm_features, axis=1, keepdims=True) + 1e-8)
        
        # Compute cosine similarity
        lda_similarity = lda_norm @ lda_norm.T
        w2v_similarity = w2v_norm @ w2v_norm.T
        
        # Combine: weighted average
        correlation = 0.5 * lda_similarity + 0.5 * w2v_similarity
        
        logger.info(f"✅ Semantic correlation matrix shape: {correlation.shape}")
        return correlation

logger.info("✅ SemanticCorrelationExtractor class defined")

2026-06-09 13:48:09,516 - INFO - ✅ SemanticCorrelationExtractor class defined


## 6. Hybrid Model with Semantic-Adjusted Edges

In [6]:
class HybridAnomalyScorer:
    """Compute hybrid anomaly scores combining graph metrics and semantic signals"""
    
    def __init__(self):
        self.hybrid_scores = None
        self.component_scores = {}
    
    def compute_graph_suspicion(self, graph_builder, review_features: np.ndarray) -> np.ndarray:
        """
        Compute graph-based suspicion scores from Random Walk anomalies
        """
        logger.info("Computing graph-based suspicion scores...")
        
        # Normalize structural features for suspicion
        scaler = StandardScaler()
        normalized_features = scaler.fit_transform(review_features)
        
        # Simple: z-score based suspicion
        graph_suspicion = np.abs(normalized_features).mean(axis=1)
        
        # Normalize to [0, 1]
        graph_suspicion = (graph_suspicion - graph_suspicion.min()) / (graph_suspicion.max() - graph_suspicion.min() + 1e-8)
        
        self.component_scores['graph'] = graph_suspicion
        logger.info(f"✅ Graph suspicion shape: {graph_suspicion.shape}")
        return graph_suspicion
    
    def adjust_edge_weights_by_semantics(self, edges: List[Tuple[int, int]], 
                                         semantic_correlation: np.ndarray,
                                         correlation_threshold: float = 0.5) -> np.ndarray:
        """
        Adjust edge weights using semantic correlation
        High correlation → lower suspicion weight
        Low correlation → higher suspicion weight
        """
        logger.info("Adjusting edge weights by semantic correlation...")
        
        edge_weights = np.ones(len(edges))
        
        for idx, (src, dst) in enumerate(edges):
            corr = semantic_correlation[src, dst]
            # Weight: 1 - correlation (inverse relationship)
            # High correlation (natural) → weight ≈ 0 (trusted)
            # Low correlation (unnatural) → weight ≈ 1 (suspicious)
            edge_weights[idx] = 1.0 - corr
        
        logger.info(f"✅ Edge weights adjusted. Mean weight: {edge_weights.mean():.4f}")
        return edge_weights
    
    def compute_semantic_suspicion(self, lda_features: np.ndarray, 
                                    lstm_features: np.ndarray,
                                    semantic_correlation: np.ndarray) -> np.ndarray:
        """
        Compute semantic-based suspicion scores
        High variance in topic distribution or low self-similarity → suspicious
        """
        logger.info("Computing semantic-based suspicion scores...")
        
        # Topic distribution entropy (high entropy = diverse topics = potentially fake)
        lda_entropy = np.array([entropy(lda_features[i]) for i in range(len(lda_features))])
        lda_suspicion = lda_entropy / (np.log(lda_features.shape[1]) + 1e-8)  # Normalize by max entropy
        
        # Self-similarity in semantic correlation (low self-sim relative to others = anomalous)
        self_similarity = np.diag(semantic_correlation)
        mean_similarity = np.mean(semantic_correlation, axis=1)
        lstm_suspicion = 1.0 - (self_similarity / (mean_similarity + 1e-8))
        lstm_suspicion = np.clip(lstm_suspicion, 0, 1)
        
        semantic_suspicion = 0.5 * lda_suspicion + 0.5 * lstm_suspicion
        
        self.component_scores['semantic'] = semantic_suspicion
        logger.info(f"✅ Semantic suspicion shape: {semantic_suspicion.shape}")
        return semantic_suspicion
    
    def compute_hybrid_score(self, graph_suspicion: np.ndarray,
                            semantic_suspicion: np.ndarray,
                            structural_suspicion: np.ndarray,
                            weights: Dict[str, float] = None) -> np.ndarray:
        """
        Compute final hybrid anomaly score combining all components
        """
        if weights is None:
            weights = {
                'graph': 0.4,
                'semantic': 0.3,
                'structural': 0.3
            }
        
        # Normalize all to [0, 1]
        graph_suspicion = (graph_suspicion - graph_suspicion.min()) / (graph_suspicion.max() - graph_suspicion.min() + 1e-8)
        semantic_suspicion = (semantic_suspicion - semantic_suspicion.min()) / (semantic_suspicion.max() - semantic_suspicion.min() + 1e-8)
        structural_suspicion = (structural_suspicion - structural_suspicion.min()) / (structural_suspicion.max() - structural_suspicion.min() + 1e-8)
        
        # Weighted combination
        hybrid_score = (
            weights['graph'] * graph_suspicion +
            weights['semantic'] * semantic_suspicion +
            weights['structural'] * structural_suspicion
        )
        
        self.hybrid_scores = hybrid_score
        logger.info(f"✅ Hybrid scores computed. Mean: {hybrid_score.mean():.4f}, Std: {hybrid_score.std():.4f}")
        return hybrid_score

logger.info("✅ HybridAnomalyScorer class defined")

2026-06-09 13:48:09,530 - INFO - ✅ HybridAnomalyScorer class defined


## 7. Build Graph and Random Walk

In [ ]:
# Build list of all edges (review-review edges for graph analysis)
logger.info("Building review graph...")

# First, check column names
logger.info(f"edges_review_review columns: {edges_review_review.columns.tolist()}")
logger.info(f"edges_review_review shape: {edges_review_review.shape}")
logger.info(f"First row:\n{edges_review_review.iloc[0]}")

review_id_map = id_mappings['review_id_map']
num_reviews = len(review_id_map)

# Use review-review edges for graph
# Dynamically find source and target columns (try multiple patterns)
src_col = None
dst_col = None

for col in edges_review_review.columns:
    col_lower = col.lower()
    if 'source' in col_lower or 'src' in col_lower:
        src_col = col
    if 'target' in col_lower or 'dest' in col_lower or 'dst' in col_lower:
        dst_col = col

if src_col is None or dst_col is None:
    raise ValueError(f"Could not find source/target columns. Available: {edges_review_review.columns.tolist()}")

logger.info(f"Using columns: {src_col} -> {dst_col}")

review_edges = []
for _, row in edges_review_review.iterrows():
    src_idx = review_id_map.get(row[src_col])
    dst_idx = review_id_map.get(row[dst_col])
    if src_idx is not None and dst_idx is not None:
        review_edges.append((src_idx, dst_idx))

logger.info(f"✅ Review graph built with {len(review_edges)} edges")
logger.info(f"✅ Number of review nodes: {num_reviews}")

2026-06-09 13:48:09,537 - INFO - Building review graph...
2026-06-09 13:48:09,537 - INFO - edges_review_review columns: ['src', 'dst', 'edge_type']
2026-06-09 13:48:09,538 - INFO - edges_review_review shape: (267837, 3)
2026-06-09 13:48:09,539 - INFO - First row:
src          85e2505bc9b89c698e28ddc703743ea0
dst          c62b91463d6dea25ee79519ff62ab352
edge_type                           same_user
Name: 0, dtype: object
2026-06-09 13:48:09,539 - INFO - Using columns: src -> dst
2026-06-09 13:48:15,788 - INFO - ✅ Review graph built with 267837 edges
2026-06-09 13:48:15,789 - INFO - ✅ Number of review nodes: 644689


2026-06-09 13:48:09,537 - INFO - Building review graph...
2026-06-09 13:48:09,537 - INFO - edges_review_review columns: ['src', 'dst', 'edge_type']
2026-06-09 13:48:09,538 - INFO - edges_review_review shape: (267837, 3)
2026-06-09 13:48:09,539 - INFO - First row:
src          85e2505bc9b89c698e28ddc703743ea0
dst          c62b91463d6dea25ee79519ff62ab352
edge_type                           same_user
Name: 0, dtype: object
2026-06-09 13:48:09,539 - INFO - Using columns: src -> dst
2026-06-09 13:48:15,788 - INFO - ✅ Review graph built with 267837 edges
2026-06-09 13:48:15,789 - INFO - ✅ Number of review nodes: 644689


: 

## 8. Build Random Walk Model

In [ ]:
# Initialize Random Walk detector with optimized parameters
# Reduce max_iter and increase tolerance for faster convergence + less memory
rw_detector = RandomWalkAnomalyDetector(num_nodes=num_reviews, damping_factor=0.85)

# Build transition probability matrix (sparse = memory efficient)
logger.info("Building transition matrix (sparse format)...")
transition_matrix = rw_detector.build_transition_matrix(review_edges)

# Compute PageRank scores with optimized parameters
logger.info("Computing PageRank (optimized for memory)...")
pagerank_scores = rw_detector.compute_pagerank(max_iter=30, tol=1e-4)  # Reduced from 100 & 1e-6

logger.info(f"✅ PageRank scores computed")
logger.info(f"   Min: {pagerank_scores.min():.6f}, Max: {pagerank_scores.max():.6f}")
logger.info(f"   Mean: {pagerank_scores.mean():.6f}, Std: {pagerank_scores.std():.6f}")

2026-06-09 13:48:15,794 - INFO - Building transition matrix (sparse format)...


## 9. Train LDA + LSTM for Semantic Extraction

In [ ]:
# Initialize semantic extractor
semantic_extractor = SemanticCorrelationExtractor(num_topics=10, embedding_dim=64)

# Prepare corpus
review_texts = reviews_clean['review_text'].fillna('').values.tolist()
dictionary, corpus, processed_texts = semantic_extractor.prepare_corpus(review_texts, min_freq=2)

# Train LDA
lda_model = semantic_extractor.train_lda(corpus, num_iterations=50)

# Extract LDA features
lda_features = semantic_extractor.extract_lda_features(corpus)

logger.info(f"✅ LDA training completed")
logger.info(f"   LDA features shape: {lda_features.shape}")

## 10. Train Word2Vec for Text Embeddings

In [ ]:
# Build Word2Vec model
semantic_extractor.build_word2vec_model(processed_texts)

# Extract Word2Vec features
word2vec_features = semantic_extractor.extract_word2vec_features(processed_texts)

logger.info(f"✅ Word2Vec training completed")
logger.info(f"   Word2Vec features shape: {word2vec_features.shape}")

## 11. Extract Word2Vec Features and Compute Semantic Correlation

In [ ]:
# Word2Vec features already extracted above
word2vec_features = semantic_extractor.lstm_features

# Compute semantic correlation
semantic_correlation = semantic_extractor.compute_semantic_correlation()

logger.info(f"✅ Semantic correlation computed")
logger.info(f"   Correlation matrix shape: {semantic_correlation.shape}")
logger.info(f"   Min: {semantic_correlation.min():.4f}, Max: {semantic_correlation.max():.4f}")
logger.info(f"   Mean: {semantic_correlation.mean():.4f}")

## 12. Compute Hybrid Anomaly Scores

In [ ]:
# Initialize hybrid scorer
hybrid_scorer = HybridAnomalyScorer()

# Extract structural features (exclude review_id, labels)
structural_features = review_features.drop(columns=['review_id', 'suspicion_score'], errors='ignore').values

# Compute component scores
graph_suspicion = hybrid_scorer.compute_graph_suspicion(None, structural_features)
semantic_suspicion = hybrid_scorer.compute_semantic_suspicion(lda_features, word2vec_features, semantic_correlation)

# Use weak labels as structural suspicion proxy
structural_suspicion = labels.astype(float)

# Compute final hybrid score
hybrid_scores = hybrid_scorer.compute_hybrid_score(
    graph_suspicion=graph_suspicion,
    semantic_suspicion=semantic_suspicion,
    structural_suspicion=structural_suspicion,
    weights={'graph': 0.4, 'semantic': 0.3, 'structural': 0.3}
)

logger.info(f"✅ Hybrid scores computed!")
logger.info(f"   Score distribution:")
logger.info(f"   Min: {hybrid_scores.min():.4f}")
logger.info(f"   Max: {hybrid_scores.max():.4f}")
logger.info(f"   Mean: {hybrid_scores.mean():.4f}")
logger.info(f"   Std: {hybrid_scores.std():.4f}")

## 13. Save Results

In [ ]:
logger.info(f"Saving anomaly detection results to {ANOMALY_DIR}...")

# Save hybrid scores
np.save(ANOMALY_DIR / "hybrid_anomaly_scores.npy", hybrid_scores)
np.save(ANOMALY_DIR / "pagerank_scores.npy", pagerank_scores)
np.save(ANOMALY_DIR / "lda_features.npy", lda_features)
np.save(ANOMALY_DIR / "word2vec_features.npy", word2vec_features)
np.save(ANOMALY_DIR / "semantic_correlation.npy", semantic_correlation)

# Save component scores
np.save(ANOMALY_DIR / "graph_suspicion.npy", hybrid_scorer.component_scores['graph'])
np.save(ANOMALY_DIR / "semantic_suspicion.npy", hybrid_scorer.component_scores['semantic'])

# Save anomaly indices
anomaly_threshold = np.percentile(hybrid_scores, 95)
anomaly_indices = np.where(hybrid_scores > anomaly_threshold)[0]
np.save(ANOMALY_DIR / "anomaly_indices.npy", anomaly_indices)

# Save metadata
metadata = {
    'num_reviews': int(num_reviews),
    'num_anomalies_detected': int(len(anomaly_indices)),
    'anomaly_percentage': float(len(anomaly_indices) / num_reviews * 100),
    'anomaly_threshold': float(anomaly_threshold),
    'hybrid_score_stats': {
        'min': float(hybrid_scores.min()),
        'max': float(hybrid_scores.max()),
        'mean': float(hybrid_scores.mean()),
        'std': float(hybrid_scores.std())
    },
    'component_weights': {
        'graph': 0.4,
        'semantic': 0.3,
        'structural': 0.3
    },
    'lda_num_topics': semantic_extractor.num_topics,
    'word2vec_embedding_dim': semantic_extractor.embedding_dim,
    'num_review_edges': len(review_edges),
    'pagerank_damping_factor': rw_detector.damping_factor
}

with open(ANOMALY_DIR / "anomaly_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"✅ All results saved to {ANOMALY_DIR}")

## 14. Visualization và Analysis

In [ ]:
import matplotlib.pyplot as plt

# Create analysis results dataframe
analysis_results = pd.DataFrame({
    'review_id': reviews_clean['review_id'].values,
    'hybrid_anomaly_score': hybrid_scores,
    'pagerank_score': pagerank_scores,
    'graph_suspicion': hybrid_scorer.component_scores['graph'],
    'semantic_suspicion': hybrid_scorer.component_scores['semantic'],
    'weak_label': labels,
    'is_anomaly_detected': (hybrid_scores > anomaly_threshold).astype(int),
    'user_id': reviews_clean['user_id'].values,
    'rating': reviews_clean['rating'].values,
    'verified_purchase': reviews_clean['verified_purchase'].values
})

# Save analysis results
analysis_results.to_parquet(ANOMALY_DIR / "anomaly_analysis_results.parquet")

# Display top anomalies
top_anomalies = analysis_results.nlargest(20, 'hybrid_anomaly_score')
logger.info("\n🔴 TOP 20 ANOMALIES:")
logger.info(top_anomalies[['review_id', 'hybrid_anomaly_score', 'pagerank_score', 'weak_label', 'rating']].to_string())

# Summary statistics
logger.info(f"\n📊 SUMMARY STATISTICS:")
logger.info(f"Total reviews: {len(analysis_results)}")
logger.info(f"Detected anomalies: {len(anomaly_indices)} ({len(anomaly_indices)/len(analysis_results)*100:.2f}%)")
logger.info(f"Weak label positives: {labels.sum()} ({labels.sum()/len(labels)*100:.2f}%)")
logger.info(f"\nAnomalies correctly detected by weak labels: {((analysis_results['is_anomaly_detected'] & (analysis_results['weak_label'] == 1)).sum())}")
logger.info(f"Weak labels correctly identified as anomalies: {((analysis_results['is_anomaly_detected'] & (analysis_results['weak_label'] == 1)).sum() / (labels.sum() + 1e-8) * 100):.2f}%")